# Treatment of the North Region Base Breast Cancer (2025)
## Data Cleaning, Preprocessing Pipeline & Breast Cancer Scope Delimitation

Predictive model of healthcare demand in the supplementary health system
(partnership with an oncology pharmaceutical company), MVP restricted to
the North region (AC, AM, AP, PA, RO, RR, TO) and to breast cancer, per the
scope defined in the project's TAPI.

### What this notebook does

Starts from the **already-merged raw** parquet of the North region
(`data/interim/north_region_ambulatorial_full_2025.parquet`, see
`02_merge_north_states.ipynb`), itself built from the 7 per-state
files, each already the result of joining TISS Ambulatorial
DET + CONS + REM by `ID_EVENTO_ATENCAO_SAUDE`. **No treatment has been
applied yet** to that raw parquet, that is the gap this notebook fills.

This notebook covers, in this order:

1. Clinical scope delimitation; applying the SBOC → TUSS crosswalk to
   restrict the base to procedures and drugs specific to breast cancer.
2. Decomposition of multivalued crosswalk columns into indicator columns
   (direct feature engineering from the clinical proxy).
3. Structural quality (key, exact duplicates).
4. Typing (string → numeric/categorical).
5. Removal of constant, near-constant, redundant columns, or columns with
   an extremely high missing rate and no relevance to the project's
   objective.
6. Treatment of the remaining missing values, column by column,
   distinguishing **structural** absence (should not be imputed) from
   **operational** absence (can be imputed) and absence in a **critical**
   field (row removed).
7. Detection and correction of outliers in the monetary/quantity columns.
8. Encoding of categorical variables.
9. Normalization of numerical variables.
10. A documented finding about the limits of the clinical proxy.
11. Export of the treated parquet + column-by-column decision dictionary.

### What this notebook does **not** do (out of scope here)

It does not define the target variable, does not select final features for
the model, and does not do a train/test split. It also does not cover lung
cancer or other regions; the scope of this deliverable is strictly the
treated North region base for breast cancer.

### Why the treatment is deliberately a "decision made today"

Every choice below (what to remove, what to impute, how to treat outliers)
is justified with the data we have right now. If the target variable
definition requires a different cut, for example, a higher
aggregation level because of statistical noise; some of these
decisions will be revisited. That is normal and expected in a real data
pipeline; that is why every decision comes with the *why*, not just the
*what*, so that revisiting a choice later does not require rebuilding the
reasoning from scratch.

### A note on language fidelity

The narrative text and every Python identifier in this notebook are in
English. The **column names** of the underlying data (`CD_PROCEDIMENTO`,
`VL_ITEM_EVENTO_INFORMADO`, `FAIXA_ETARIA`, etc.) and the **categorical
values found inside the data** (e.g. `"Feminino"`, `"GRANDE"`, the age-group
labels) are kept exactly as published by ANS; these are the official field
names and recorded values of the source dataset, not narrative text, and
translating them would break traceability back to the original TISS/TUSS
documentation and to the data itself.

## Phase 0 — Setup

In [1]:
import json
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (9, 5)

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = REPO_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
REFERENCE_DIR = DATA_DIR / "reference"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = INTERIM_DIR / "north_region_ambulatorial_full_2025.parquet"
CROSSWALK_PATH = REFERENCE_DIR / "tuss_sboc_crosswalk_breast_cancer.csv"
OUTPUT_PATH = PROCESSED_DIR / "north_region_mama_2025_data_en.parquet"
DECISIONS_DICTIONARY_PATH = PROCESSED_DIR / "north_region_mama_2025_decisions_dictionary.csv"

required_inputs = {
    RAW_PATH: "02_merge_north_states.ipynb",
    CROSSWALK_PATH: "03_tuss_sboc_crosswalk.ipynb",
}
missing_inputs = [(path, producer) for path, producer in required_inputs.items() if not path.is_file()]
if missing_inputs:
    details = "\n".join(f"- Run {producer}; expected: {path}" for path, producer in missing_inputs)
    raise FileNotFoundError("Missing upstream artifact(s) required by notebook 04:\n" + details)

print(f"Raw parquet: {RAW_PATH}")
print(f"Crosswalk: {CROSSWALK_PATH}")

Raw parquet: C:\Users\Inteli\Desktop\g02\data\interim\north_region_ambulatorial_full_2025.parquet
Crosswalk: C:\Users\Inteli\Desktop\g02\data\reference\tuss_sboc_crosswalk_breast_cancer.csv


## Phase 1 — Clinical scope delimitation: SBOC → TUSS crosswalk

The raw North region parquet has **almost 60 million rows** (one per
billed item/procedure, across any specialty) and covers the entire TISS
Ambulatorial dataset of the 7 states in 2025 — not just breast cancer. The
first treatment step, before any cleaning, is to restrict the base to the
procedures and drugs clinically related to breast cancer.

**Why filter first, before typing/cleaning?**

**Practical memory reason**: filtering first drastically reduces the
   volume before any heavier operation (null detection, outliers,
   encoding). This avoids repeating the RAM-overflow problem already seen
   in the single-state pilot, only now multiplied by 7 states.

### How the filter works

We reuse the crosswalk already built and documented in a list of TUSS codes
(`tuss_table` + `tuss_code`, corresponding to `CD_TABELA_REFERENCIA` +
`CD_PROCEDIMENTO` in the raw data) obtained by matching the clinical terms
of the 4 guidelines from the Brazilian Society of Clinical Oncology (SBOC)
for breast cancer; Staging (v9), Neoadjuvant (v6), Adjuvant (v6) and
Advanced Disease (v12) (1), against the real TUSS tables of drugs (table
20) and procedures (table 22/63) (2).

The crosswalk classifies every code found into two specificity tiers:

- **`specific`**: procedures/drugs that only make clinical sense associated
  with breast cancer (e.g. mammography, breast biopsy, mastectomy,
  trastuzumab) or with general oncology treatment lines cited in the breast
  cancer guidelines (e.g. doxorubicin, cyclophosphamide).
- **`generic_staging`**: broad oncology staging exams (chest/abdomen CT,
  brain MRI, bone scintigraphy, PET, bone densitometry), used across
  dozens of other malignancies, with no CID (diagnosis code) field
  available in TISS Ambulatorial to confirm a breast-specific relationship
  (Ambulatorial, unlike Hospitalar, has no CID field in any of its files).

**Decision**: use **only** the `specific` codes as the inclusion filter,
discarding the `generic_staging` ones entirely, not just as the primary
filter, but also as a supplementary signal. Reason: these are exams used
across dozens of other oncology conditions; including them, even as
metadata on already-selected events, would risk introducing ambiguity into
a decision (what counts as "breast-cancer-related") that needs to be as
defensible as possible, since it is the central proxy variable of the
entire project. We keep this decision identical to
the one used in the single-state pilot that first established this
crosswalk, for methodological consistency between the two stages of the
project.

In [2]:
crosswalk_full = pd.read_csv(CROSSWALK_PATH, dtype=str)
crosswalk_full = crosswalk_full[crosswalk_full["category"].isin(["drug", "procedure", "procedure_group"])]

crosswalk = crosswalk_full[crosswalk_full["specificity"] == "specific"].copy()
print(f"Breast-cancer-specific TUSS codes (inclusion filter): {len(crosswalk)} rows, "
      f"{crosswalk[['tuss_table', 'tuss_code']].drop_duplicates().shape[0]} unique (table, code) pairs")
print(f"Generic staging codes discarded: {(crosswalk_full['specificity'] == 'generic_staging').sum()}")

# lookup table: for each (table, code), which categories/treatment lines/
# subtypes from the SBOC guidelines it represents. When the same code
# appears in more than one treatment line or subtype, the values are
# concatenated with "|" -- handled in Phase 2 (decomposition).
crosswalk_lookup = (
    crosswalk.groupby(["tuss_table", "tuss_code"])
    .agg({
        "category": lambda x: "|".join(sorted(set(x))),
        "treatment_line": lambda x: "|".join(sorted(set(x))),
        "subtype": lambda x: "|".join(sorted(set(x))),
        "search_term": lambda x: "|".join(sorted(set(x))),
    })
    .reset_index()
)
print(f"\nUnique codes in the lookup table: {len(crosswalk_lookup)}")
crosswalk_lookup.head()

Breast-cancer-specific TUSS codes (inclusion filter): 385 rows, 384 unique (table, code) pairs
Generic staging codes discarded: 12

Unique codes in the lookup table: 384


,tuss_table,tuss_code,category,treatment_line,subtype,search_term
0,20,90002032,drug,"neoadjuvant,adjuvant,advanced",general,carboplatina
1,20,90002040,drug,"neoadjuvant,adjuvant,advanced",general,carboplatina
2,20,90002067,drug,advanced,general,cisplatina
3,20,90002083,drug,advanced,general,cisplatina
4,20,90002130,drug,"neoadjuvant,advanced",general,epirrubicina


### Applying the filter to the raw parquet (batched reading)

Reading the nearly 60 million rows entirely into a pandas `DataFrame`
before filtering would require several GB of RAM just for data that will
be discarded next, the same memory problem documented in the per-state
ingestion notebook. Instead, we read the parquet **row-group by row-group**
via PyArrow (the native partial-read granularity of the parquet format),
apply the filter to each chunk, and only materialize into pandas what
survives the filter, which, as we will see, is a very small fraction of
the raw base.

In [3]:
valid_codes = set(zip(crosswalk["tuss_table"], crosswalk["tuss_code"]))
valid_key = pa.array([f"{t}|{c}" for t, c in valid_codes], type=pa.string())

source = pq.ParquetFile(RAW_PATH)
print(f"Raw parquet: {source.metadata.num_row_groups} row groups, {source.metadata.num_rows:,} rows")

filtered_parts = []
total_read = 0

for i in range(source.metadata.num_row_groups):
    table = source.read_row_group(i)
    total_read += table.num_rows

    key = pc.binary_join_element_wise(
        table.column("CD_TABELA_REFERENCIA").cast(pa.string()),
        table.column("CD_PROCEDIMENTO").cast(pa.string()),
        "|",
    )
    mask = pc.is_in(key, value_set=valid_key)
    filtered_table = table.filter(mask)
    if filtered_table.num_rows > 0:
        filtered_parts.append(filtered_table)

    if (i + 1) % 20 == 0 or i == source.metadata.num_row_groups - 1:
        filtered_so_far = sum(t.num_rows for t in filtered_parts)
        print(f"  row group {i + 1}/{source.metadata.num_row_groups}: "
              f"read={total_read:,} | filtered so far={filtered_so_far:,}")

df = pa.concat_tables(filtered_parts).to_pandas()
del filtered_parts

print(f"\nTotal rows in the raw base: {total_read:,}")
print(f"Total rows related to breast cancer: {len(df):,} ({len(df) / total_read * 100:.4f}%)")

Raw parquet: 117 row groups, 59,913,304 rows


  row group 20/117: read=6,793,465 | filtered so far=34,829


  row group 40/117: read=18,885,995 | filtered so far=96,357


  row group 60/117: read=30,836,613 | filtered so far=164,170


  row group 80/117: read=47,379,535 | filtered so far=258,348


  row group 100/117: read=54,957,051 | filtered so far=299,720


  row group 117/117: read=59,913,304 | filtered so far=324,469

Total rows in the raw base: 59,913,304
Total rows related to breast cancer: 324,469 (0.5416%)


**Real result obtained**: of the **59,913,304** rows in the raw North
region base, **324,469** (0.5416%) correspond to some breast-cancer-specific
TUSS code. That is a small share in proportion as expected, given that the
raw base covers the entire TISS Ambulatorial dataset (every specialty)
across the 7 states, and we are isolating a single disease. This volume
(~324 thousand rows, ~304 thousand unique events) is perfectly manageable
in memory for the rest of this notebook.

### Enriching with the crosswalk columns

We merge `crosswalk_lookup` back in to bring, for every row of the filtered
base, the category (drug/procedure), the SBOC treatment line(s)
(staging/neoadjuvant/adjuvant/advanced) and the clinical subtype(s) (HER2+,
HR+/HER2-, triple-negative, BRCA-mutated, general, etc.) associated with
the billed code. These columns are the central clinical proxy of the
project, with no lab result available in TISS, they are the
only information available on "what type of potential patient" that event
might represent.

In [4]:
df = df.merge(
    crosswalk_lookup,
    left_on=["CD_TABELA_REFERENCIA", "CD_PROCEDIMENTO"],
    right_on=["tuss_table", "tuss_code"],
    how="left",
).drop(columns=["tuss_table", "tuss_code"])

assert df["category"].isna().sum() == 0, "Row with no crosswalk match -- should not happen after filtering."
print("Base enriched with crosswalk columns:", df.shape)
df[["CD_TABELA_REFERENCIA", "CD_PROCEDIMENTO", "category", "treatment_line", "subtype"]].head()

Base enriched with crosswalk columns: (324469, 37)


,CD_TABELA_REFERENCIA,CD_PROCEDIMENTO,category,treatment_line,subtype
0,22,40901114,procedure,"staging,neoadjuvant",general
1,22,40901114,procedure,"staging,neoadjuvant",general
2,22,40808041,procedure,"staging,neoadjuvant,adjuvant",general
3,22,40808041,procedure,"staging,neoadjuvant,adjuvant",general
4,22,40808041,procedure,"staging,neoadjuvant,adjuvant",general


## Phase 2 — Decomposing multivalued columns into indicator columns

Both `treatment_line` and `subtype` (from the crosswalk) and
`CD_MODELO_REMUNERACAO` (from REM, already concatenated with `"|"` when an
event has more than one reimbursement model, see `build_am_base_2025.py`)
can hold more than one value in the same cell. Two different separators
show up, and both mean the same thing (more than one value applies): `"|"`
joins values that came from different crosswalk rows matched to the same
TUSS code (added by the `crosswalk_lookup` aggregation in Phase 1), while
`","` already joins multiple treatment lines listed together on a single
crosswalk row, straight from the SBOC guidelines (e.g. paclitaxel is cited
as `"neoadjuvant,adjuvant,advanced"` on one single guideline entry, since
it is used across all three lines).

Keeping these columns as concatenated strings has two problems: (1) they
cannot be used directly in any statistical or machine-learning model
without reprocessing, and (2) they hide the real granularity of the
information (an event tagged `"neoadjuvant,adjuvant"` is not a third
category, it is both at once). The fix is
**normalization into adjacent columns**: one binary indicator column per
atomic value, instead of a single multivalued text column, which means
splitting on both separators, not just `"|"`.

In [5]:
def split_multivalued_cell(cell):
    """A single cell can hold more than one value in two different ways:
    '|' joins values that came from different crosswalk rows for the same
    TUSS code (added by the crosswalk_lookup aggregation above), while ','
    joins multiple treatment lines already listed together on a single
    crosswalk row (e.g. a drug cited for 'neoadjuvant,adjuvant,advanced' in
    the guidelines). Both mean the same thing here -- more than one value
    applies -- so both are split to get the real atomic values."""
    if pd.isna(cell):
        return []
    return [token for token in cell.replace(",", "|").split("|") if token]


def decompose_multivalued(series, prefix):
    """Takes a string series possibly holding 'a|b,c' and returns a
    DataFrame with one binary (0/1) indicator column per unique atomic value
    found across the whole series (see split_multivalued_cell above)."""
    unique_values = sorted(set(
        v for cell in series.dropna() for v in split_multivalued_cell(cell)
    ))
    output = pd.DataFrame(index=series.index)
    for v in unique_values:
        column_name = f"{prefix}_{v}".upper()
        column_name = (
            unicodedata.normalize("NFKD", column_name)
            .encode("ascii", "ignore").decode("ascii")
            .replace(" ", "_").replace("-", "_").replace("/", "_").replace("+", "POS")
        )
        output[column_name] = series.apply(lambda cell: int(v in split_multivalued_cell(cell)))
    return output

treatment_line_dummies = decompose_multivalued(df["treatment_line"], "IND_LINE")
subtype_dummies = decompose_multivalued(df["subtype"], "IND_SUBTYPE")
reimbursement_model_dummies = decompose_multivalued(df["CD_MODELO_REMUNERACAO"], "IND_REIMBURSEMENT_MODEL")

print("Treatment-line indicator columns:", list(treatment_line_dummies.columns))
print("Clinical subtype indicator columns:", list(subtype_dummies.columns))
print("Reimbursement-model indicator columns:", list(reimbursement_model_dummies.columns))

df = pd.concat([df, treatment_line_dummies, subtype_dummies, reimbursement_model_dummies], axis=1)
print(f"\nShape after decomposition: {df.shape}")

Treatment-line indicator columns: ['IND_LINE_ADJUVANT', 'IND_LINE_ADVANCED', 'IND_LINE_NEOADJUVANT', 'IND_LINE_STAGING']
Clinical subtype indicator columns: ['IND_SUBTYPE_BRCA_MUTATED', 'IND_SUBTYPE_HER2_UNDETERMINED', 'IND_SUBTYPE_HER2POS', 'IND_SUBTYPE_HER2POS_HER2_LOW', 'IND_SUBTYPE_HRPOS_HER2_', 'IND_SUBTYPE_NTRKPOS', 'IND_SUBTYPE_GENERAL', 'IND_SUBTYPE_TRIPLE_NEGATIVE']
Reimbursement-model indicator columns: ['IND_REIMBURSEMENT_MODEL_01', 'IND_REIMBURSEMENT_MODEL_02', 'IND_REIMBURSEMENT_MODEL_04']

Shape after decomposition: (324469, 52)


**Why not drop the original `treatment_line`/`subtype` columns yet:**
they remain useful in human-readable form for the next phases and for Phase 12 of this notebook (the finding on the proxy's
limits). We only drop them at export time, if the team decides the
indicator columns are enough on their own, we leave that decision
recorded, not made, because it depends on how we end up designing the
final feature set.

`CD_MODELO_REMUNERACAO` is kept as-is (string, potentially multivalued) for
provenance/audit purposes, with the `IND_REIMBURSEMENT_MODEL_*` columns as
the analysis-ready version.

## Phase 3 — Structural quality: unique key and exact duplicates

Before typing or treating any column, we confirm the event key is never
null (repeating the assertion here costs
nothing and guards against regression) and check for 100%-duplicate rows,
a diagnostic the per-state ingestion notebook deliberately left for this
stage.

In [6]:
assert df["ID_EVENTO_ATENCAO_SAUDE"].isna().sum() == 0, "There are events with no key -- investigate before proceeding."

duplicates = df.duplicated()
print(f"Fully duplicated rows (every column identical): {duplicates.sum():,} "
      f"({duplicates.mean() * 100:.4f}%)")

df = df[~duplicates].reset_index(drop=True)
print(f"Shape after removing exact duplicates: {df.shape}")

Fully duplicated rows (every column identical): 203 (0.0626%)


Shape after removing exact duplicates: (324266, 52)


**Decision**: remove them, keeping the first occurrence. In the North
region breast-cancer subset, that is **203 rows (0.063%)**. Two
explanations are plausible for a 100%-identical duplicate (same event key,
same procedure, same value, even the same month): a resubmission/retransmission
of the TISS file by the operator (a known artifact of the ANS collection
process), or the same procedure genuinely billed twice within the same
event. Since the date granularity available in Ambulatorial is monthly
(`DT_REALIZACAO` in AAAA-MM format, with no day), there is no way to tell
the two hypotheses apart from the data alone. We chose to remove because,
for the project's objective (estimating potential patient volume), counting
the same row twice artificially inflates volume, the risk of undercounting
a rare case of legitimate duplicate billing is smaller than the risk of
overestimating demand because of a resubmission artifact, and the
percentage involved (0.063%) is low enough that either error has negligible
impact on the final result.

## Phase 4 — Typing: from string to numeric/categorical

The raw parquet keeps **every** column as string, exactly as it came from
the original ANS CSV (a deliberate decision documented in the ingestion
notebook, typing would only make sense after filtering, to avoid repeating
the logic 7 times). Now that the base is filtered and small, we convert
each column to its correct analytical type, following the provenance
dictionary defined.

In [7]:
# --- monetary/quantity columns: comma-decimal string -> float ---
NUMERIC_COLUMNS = [
    "QT_ITEM_EVENTO_INFORMADO", "VL_ITEM_EVENTO_INFORMADO",
    "VL_ITEM_PAGO_FORNECEDOR",
]
for c in NUMERIC_COLUMNS:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

# VL_TOTAL_REMUNERACAO and N_LINHAS_REM already arrive as float/double from
# the raw parquet (converted in Phase 3 of the ingestion notebook, see
# Phase 2 there).
print(df[NUMERIC_COLUMNS + ["VL_TOTAL_REMUNERACAO", "N_LINHAS_REM"]].dtypes)

QT_ITEM_EVENTO_INFORMADO    float64
VL_ITEM_EVENTO_INFORMADO    float64
VL_ITEM_PAGO_FORNECEDOR     float64
VL_TOTAL_REMUNERACAO        float64
N_LINHAS_REM                float64
dtype: object


In [8]:
# --- date: DT_REALIZACAO and MES arrive as 'AAAA-MM' / 'AAAAMM' -- converted
# to a monthly period (Timestamp on the first day of the month). We keep
# monthly, not daily, granularity because that is the real granularity
# available in TISS Ambulatorial (there is no day field in the raw data).
df["DT_REALIZACAO"] = pd.to_datetime(df["DT_REALIZACAO"], format="%Y-%m", errors="coerce")
df["MES"] = pd.to_datetime(df["MES"], format="%Y%m", errors="coerce")

# --- nominal categoricals (no natural order) ---
NOMINAL_CATEGORICAL_COLUMNS = [
    "UF_PRESTADOR", "CD_PROCEDIMENTO", "CD_TABELA_REFERENCIA", "UNIDADE_MEDIDA",
    "IND_PACOTE", "IND_TABELA_PROPRIA", "ID_PLANO", "SEXO",
    "CD_MUNICIPIO_BENEFICIARIO", "CD_MODALIDADE", "NM_MODALIDADE",
    "CD_MUNICIPIO_PRESTADOR", "CD_CARATER_ATENDIMENTO", "TIPO_ATENDIMENTO",
    "REGIME_ATENDIMENTO", "CD_MOTIVO_SAIDA", "CBO", "TIPO_CONSULTA",
    "SAUDE_OCUPACIONAL", "IND_ACIDENTE_DOENCA", "LG_VALOR_PREESTABELECIDO",
    "CD_MODELO_REMUNERACAO", "UF", "category",
]
for c in NOMINAL_CATEGORICAL_COLUMNS:
    df[c] = df[c].astype("category")

print(f"{len(NUMERIC_COLUMNS) + 2} numeric/temporal columns and "
      f"{len(NOMINAL_CATEGORICAL_COLUMNS)} nominal categorical columns typed.")

5 numeric/temporal columns and 24 nominal categorical columns typed.


`FAIXA_ETARIA` and `PORTE` are **not** included in the nominal typing above:
both have a natural order (age group increases, operator size increases)
and will become explicit ordinal categories in Phase 10 (Encoding), after
the invalid `FAIXA_ETARIA` values are handled in Phase 6. Doing this now
would be premature, we first need to decide what to do with the
`"Informada Incorretamente"` and `"Nao identificado"` values.

## Phase 5 — Columns to drop

We are removing columns with an
extremely high missing-value rate or no relevance to the project's
objective, as long as the decision is documented. Before deciding what to
impute, we look at the real missing rate and cardinality of every column in
the already-filtered subset (not the raw base, the missing rate changes
substantially after the clinical cut, so deciding on the wrong data would
lead to the wrong decision).

In [9]:
diagnostics = pd.DataFrame({
    "pct_null": (df.isna().sum() / len(df) * 100).round(2),
    "n_unique": df.nunique(),
}).sort_values("pct_null", ascending=False)
diagnostics

,pct_null,n_unique
SAUDE_OCUPACIONAL,99.31,3
UNIDADE_MEDIDA,93.87,17
CD_MOTIVO_SAIDA,69.93,11
TIPO_CONSULTA,67.68,4
CBO,44.21,85
VL_TOTAL_REMUNERACAO,6.17,27570
N_LINHAS_REM,6.17,2
CD_MODELO_REMUNERACAO,6.17,4
QT_ITEM_EVENTO_INFORMADO,5.46,436
VL_ITEM_EVENTO_INFORMADO,5.46,6514


### Column-removal decisions, one by one

| Column | Real % null | Reason for removal |
|---|---|---|
| `IND_TABELA_PROPRIA` | 0% null, but **1 unique value** (`'0'`) across the entire base | Zero variance — no row in the breast-cancer subset uses the operator's own custom table. A constant column carries no discriminative information for any analysis or model. |
| `LG_VALOR_PREESTABELECIDO` | 0% null, but **99.999%** the same value (`'0'`; only 4 rows out of 324 thousand are `'1'`) | Near-constant — practically the same justification as above. The 4 exceptions do not support a usable feature. |
| `SAUDE_OCUPACIONAL` | **99.31%** | Occupational-health field, almost never applicable to outpatient oncology care. With <1% fill rate, there is not enough data density for any analytical use. |
| `CD_MOTIVO_SAIDA` | **69.93%** | High missing rate **and** low conceptual relevance to the project's objective (estimating potential patient volume by operator/region/service) — the field describes the reason an episode was closed, which is closer to care outcome than to demand volume. Unlike `UNIDADE_MEDIDA`/`TIPO_CONSULTA` (Phase 8), the absence here is not fully explained by another column present in the base, so there is no clean way to impute without risking bias. |
| `CD_MODALIDADE` | 0% null | **Redundant**: we verified that `CD_MODALIDADE` and `NM_MODALIDADE` have an exact 1:1 correspondence (each code maps to exactly one name, with no exception, across the 6 observed values). We keep `NM_MODALIDADE` (human-readable — "Cooperativa Medica", "Autogestao", "Medicina De Grupo", etc.) and drop the code, which adds no new information and is less interpretable for whoever reads the final report (Risk R05 of the TAPI, about accessible language for non-technical users). |

Note that `UNIDADE_MEDIDA` (93.87% null) and `TIPO_CONSULTA` (67.70% null)
are **not** on this list, despite the high percentage, Phase 8 shows why
their absence is structural and explained by another column, which changes
the decision from "remove" to "keep and impute an explicit label".

In [10]:
COLUMNS_TO_DROP = [
    "IND_TABELA_PROPRIA", "LG_VALOR_PREESTABELECIDO",
    "SAUDE_OCUPACIONAL", "CD_MOTIVO_SAIDA", "CD_MODALIDADE",
]
df = df.drop(columns=COLUMNS_TO_DROP)
print(f"Columns dropped: {COLUMNS_TO_DROP}")
print(f"Shape after removal: {df.shape}")

Columns dropped: ['IND_TABELA_PROPRIA', 'LG_VALOR_PREESTABELECIDO', 'SAUDE_OCUPACIONAL', 'CD_MOTIVO_SAIDA', 'CD_MODALIDADE']
Shape after removal: (324266, 47)


## Phase 6 — Data-quality finding: the 210 records with invalid demographics

Before deciding row by row what to do about missing values, one specific
finding deserves its own spotlight because it changes how we treat three
columns at once. While investigating `FAIXA_ETARIA`, we found two values
that are not real age groups: `"Informada Incorretamente"` (incorrectly
reported) and `"Nao identificado"` (not identified), these are error
labels from ANS's own source system, not valid age categories.

In [11]:
print(df["FAIXA_ETARIA"].value_counts())

FAIXA_ETARIA
40 a 49 anos                104085
50 a 59 anos                 68318
30 a 39 anos                 47905
60 a 69 anos                 42516
20 a 29 anos                 22022
70 a 79 anos                 19945
80 anos ou mais               5549
15 a 19 anos                  4977
5 a 9 anos                    3530
1 a 4 anos                    2657
10 a 14 anos                  2195
Menos de 1 ano                 357
Informada Incorretamente       210
Name: count, dtype: int64


In [12]:
invalid_age_group = df["FAIXA_ETARIA"].isin(["Informada Incorretamente", "Nao identificado"])
null_sex = df["SEXO"].isna()
null_municipality = df["CD_MUNICIPIO_BENEFICIARIO"].isna()

print(f"Invalid FAIXA_ETARIA: {invalid_age_group.sum()}")
print(f"Null SEXO: {null_sex.sum()}")
print(f"Null CD_MUNICIPIO_BENEFICIARIO: {null_municipality.sum()}")
print(f"Do all three conditions happen on exactly the same rows? "
      f"{(invalid_age_group.values == null_sex.values).all() and (invalid_age_group.values == null_municipality.values).all()}")
print(f"Union of the three conditions: {(invalid_age_group | null_sex | null_municipality).sum()} rows "
      f"({(invalid_age_group | null_sex | null_municipality).mean() * 100:.4f}%)")

Invalid FAIXA_ETARIA: 210
Null SEXO: 210
Null CD_MUNICIPIO_BENEFICIARIO: 210
Do all three conditions happen on exactly the same rows? True
Union of the three conditions: 210 rows (0.0648%)


**Finding**: the 210 rows with an invalid `FAIXA_ETARIA` are **exactly the
same** 210 rows with a null `SEXO` and **exactly the same** 210 rows with a
null `CD_MUNICIPIO_BENEFICIARIO`, this is not a statistical coincidence,
it is a single cluster of events whose source `CONS` record came in with
the entire demographic block either corrupted or unfilled by the operator.
This is evidence in favor of a source-side fill failure (the operator did
not report the field), rather than a problem in our own join pipeline.

**Decision**: remove these 210 rows (0.065% of the filtered base), instead
of imputing. Sex, age group, and municipality of residence are exactly the
core demographic/geographic variables of the demand model (the TAPI's own
problem definition uses region, age group, and sex as segmentation axes);
imputing a value for them would mean inventing a patient characteristic we
have no basis to estimate. With such a low volume (210 out of 324
thousand), the removal has negligible statistical impact.

In [13]:
invalid_rows = invalid_age_group | null_sex | null_municipality
n_invalid_rows = invalid_rows.sum()
df = df[~invalid_rows].reset_index(drop=True)
print(f"Shape after removing the {n_invalid_rows:,} rows with invalid demographics: {df.shape}")

Shape after removing the 210 rows with invalid demographics: (324056, 47)


## Phase 7 — Age-implausibility filter: pediatric/adolescent records under the breast-cancer proxy

The TUSS/SBOC crosswalk applied in Phase 1 delimits this base by billing
codes, not by a confirmed diagnosis: records billed under a broad,
non-breast-specific chemotherapy code (`subtype == "general"`) can belong to
a patient being treated for a different cancer that happens to share the same
drug. Phase 12 measures that imprecision along the sex axis. Here we address
it along an axis that has to be settled before any treatment phase runs: age. `FAIXA_ETARIA` in
this already-filtered breast-cancer base still contains brackets from
`"Menos de 1 ano"` through `"15 a 19 anos"`, i.e., patients under 20 years
old.

This is not a rare subgroup of breast cancer, it is a clinical
near-impossibility. Breast cancer is a disease of adulthood: incidence is
negligible before age 20 and rises sharply after 40, which is exactly why
INCA's national incidence estimates [3] (already used as
qualitative epidemiological context in this project, see Section 4.1 /
Risk R08) report no meaningful case volume for that age range, and why the
four SBOC guidelines behind our own crosswalk (Phase 1) are staging and
treatment protocols written for an adult disease. Pediatric oncology
treats a different set of cancers, acute leukemias, lymphomas, and
sarcomas being the most common in this age range, using several of the
same broad cytotoxic drugs (doxorubicin, cyclophosphamide) our crosswalk
had to include for adult breast-cancer chemotherapy. With no diagnosis
(CID) field anywhere in TISS Ambulatorial, a record billing one of these
drugs to a 6-year-old is, for this pipeline, indistinguishable from a real
breast-cancer event unless we apply this kind of outside clinical
reasoning explicitly.

**Why this filter runs here, before any treatment phase:** every phase that
follows computes statistics across rows. The outlier bounds of Phase 9 are a
mean and a standard deviation in log space, and the min-max scaling of
Phase 11 reads the extremes of each column. Rows we have already concluded
are not breast-cancer events must not contribute to those quantities, or the
limits applied to the base we ship would be partly defined by records that
are not in it. Filtering here, immediately after the demographic-quality work
of Phase 6, keeps every downstream statistic computed on exactly the
population the final base represents.

We now quantify how much of the base this affects, and check whether the same
three signals that Phase 12 later uses to characterize the proxy (subtype
composition, proxy specificity, sex distribution) point the same way here.

In [14]:
PEDIATRIC_ADOLESCENT_AGE_GROUPS = [
    "Menos de 1 ano", "1 a 4 anos", "5 a 9 anos", "10 a 14 anos", "15 a 19 anos",
]
under_20 = df["FAIXA_ETARIA"].isin(PEDIATRIC_ADOLESCENT_AGE_GROUPS)

print(df.loc[under_20, "FAIXA_ETARIA"].value_counts().reindex(PEDIATRIC_ADOLESCENT_AGE_GROUPS))
print(f"\nTotal rows under 20 years old: {under_20.sum():,} ({under_20.mean() * 100:.4f}% of the base)")

FAIXA_ETARIA
Menos de 1 ano     357
1 a 4 anos        2657
5 a 9 anos        3530
10 a 14 anos      2195
15 a 19 anos      4977
Name: count, dtype: int64

Total rows under 20 years old: 13,716 (4.2326% of the base)


In [15]:
print("Subtype composition among under-20 records:")
print(df.loc[under_20, "subtype"].value_counts())

# IND_HIGH_SPECIFICITY_PROXY (Phase 12) is a pure function of `subtype`, so the
# same signal is available here, before encoding has produced df_encoded.
high_specificity = df["subtype"] != "general"
proxy_rate_under_20 = high_specificity[under_20].mean() * 100
proxy_rate_overall = high_specificity.mean() * 100
male_share_under_20 = (df.loc[under_20, "SEXO"] == "Masculino").mean() * 100

print(f"\nHigh-specificity proxy rate, under-20 records: {proxy_rate_under_20:.4f}%")
print(f"High-specificity proxy rate, full base:         {proxy_rate_overall:.4f}%")
print(f"Male share among under-20 records:               {male_share_under_20:.2f}%")


Subtype composition among under-20 records:
subtype
general      13707
HR+/HER2-        9
Name: count, dtype: int64

High-specificity proxy rate, under-20 records: 0.0656%
High-specificity proxy rate, full base:         4.6600%
Male share among under-20 records:               41.72%


**Real result obtained**: **13,716** rows (**4.2326%** of the 324,056-row
base at this point) fall in a `FAIXA_ETARIA` bracket under 20 years old:
357 "Menos de 1 ano", 2,657 "1 a 4 anos", 3,530 "5 a 9 anos", 2,195
"10 a 14 anos", and 4,977 "15 a 19 anos". All three signals that
Phase 12 later uses to characterize the proxy point the same way for this
group, only more extreme:

- **Subtype**: **13,707 of the 13,716 rows (99.93%)** carry the
  non-specific `"general"` subtype; only **9 rows** fall under a
  breast-specific subtype (`HR+/HER2-`).
- **Proxy specificity**: the high-specificity proxy rate inside this group
  is **0.0656%**, against **4.66%** for the base as a whole at this point
  (the 324,056 rows this filter is about to reduce), roughly a **71x** gap.
- **Sex distribution**: **41.72%** of these under-20 records are male,
  more than 40 times the ~1% incidence expected for male breast cancer even
  among adults, where this same distortion is a
  limitation of the proxy that Phase 12 documents in detail.

Three independent signals agreeing on the same conclusion is strong
evidence, not coincidence: these records are almost certainly other
pediatric/adolescent malignancies (leukemias, lymphomas, sarcomas) billed
with the same broad cytotoxic drugs our crosswalk had to include for adult
breast-cancer treatment, combined with the complete absence of a CID field
to tell the two apart. Under current medical knowledge, breast cancer in
this age range is not a small real segment to be preserved, it is a
clinical impossibility at the volume observed here.

**Decision**: remove these 13,716 rows (4.2326% of the base). This differs
from both precedents already set in this notebook:

- Unlike the monetary outliers of Phase 9, which we deliberately do
  **not** discard because they represented genuinely expensive but real
  breast-cancer treatments (the project's central commercial signal), there
  is no equivalent upside here: no plausible share of the region's real
  breast-cancer demand comes from patients under 20.
- Unlike the `"general"` subtype that Phase 12 flags rather than removes, where
  the same broad-drug ambiguity still leaves a **majority** of genuine
  breast-cancer cases inside the group, the age criterion alone already
  isolates a group that is over 99.9% non-breast-cancer by the crosswalk's
  own subtype classification. A partial confidence flag would still leave
  clinically impossible records inside the modeling base with no principled
  way to weight them down to zero, so outright removal, not flagging, is
  the consistent choice here.

In [16]:
df = df[~under_20].reset_index(drop=True)

print(f"Shape after removing the {under_20.sum():,} under-20 rows: {df.shape}")


Shape after removing the 13,716 under-20 rows: (310340, 47)


## Phase 8 — Remaining missing values: structural vs. operational

With the data already filtered, duplicates removed, irrelevant columns
dropped, and the 210 invalid-demographics rows out, we recompute the null
rate to decide what is left.

In [17]:
remaining_nulls = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
remaining_nulls[remaining_nulls > 0].round(2)

UNIDADE_MEDIDA              93.61
TIPO_CONSULTA               68.23
CBO                         44.11
VL_TOTAL_REMUNERACAO         6.35
CD_MODELO_REMUNERACAO        6.35
N_LINHAS_REM                 6.35
QT_ITEM_EVENTO_INFORMADO     4.76
VL_ITEM_PAGO_FORNECEDOR      4.76
VL_ITEM_EVENTO_INFORMADO     4.76
REGIME_ATENDIMENTO           0.46
TIPO_ATENDIMENTO             0.11
CD_CARATER_ATENDIMENTO       0.08
IND_ACIDENTE_DOENCA          0.07
dtype: float64

### Group 1 — Structural absence explained by another column (do not impute with a generic value; impute an explicit "not applicable" label)

**`UNIDADE_MEDIDA`**: we verified it is null in **100%** of the rows whose
`CD_TABELA_REFERENCIA` is `22` (procedure) or `63` (procedure group), and
filled in **100%** of the rows with `CD_TABELA_REFERENCIA = 20` (drug).
This makes clinical sense: unit of measure (milligram, ampoule, tablet...)
only exists for a drug, not for a procedure. It is not "missing data" in
the sense of an error, it is a field that **does not apply** to most rows
by definition. We impute an explicit label
(`"Not applicable (procedure)"`) instead of leaving it null, so the meaning
stays recorded in the column itself.

**`TIPO_CONSULTA`**: null in 67.70% of rows. Per the definition of TUSS
table 52, this field only applies to events that are actually a
consultation, not to a chemotherapy drug administration or to an exam.
Same logic: we impute `"Not applicable (not a consultation)"`.

In [18]:
df["UNIDADE_MEDIDA"] = df["UNIDADE_MEDIDA"].cat.add_categories(["NOT_APPLICABLE_PROCEDURE"])
df["UNIDADE_MEDIDA"] = df["UNIDADE_MEDIDA"].fillna("NOT_APPLICABLE_PROCEDURE")

df["TIPO_CONSULTA"] = df["TIPO_CONSULTA"].cat.add_categories(["NOT_APPLICABLE_NOT_A_CONSULTATION"])
df["TIPO_CONSULTA"] = df["TIPO_CONSULTA"].fillna("NOT_APPLICABLE_NOT_A_CONSULTATION")

print("Remaining UNIDADE_MEDIDA nulls:", df["UNIDADE_MEDIDA"].isna().sum())
print("Remaining TIPO_CONSULTA nulls:", df["TIPO_CONSULTA"].isna().sum())

Remaining UNIDADE_MEDIDA nulls: 0
Remaining TIPO_CONSULTA nulls: 0


### Group 2 — Structural absence tied to REM (ANS processing lag; keep null, flag it)

`VL_TOTAL_REMUNERACAO`, `CD_MODELO_REMUNERACAO`/`IND_REIMBURSEMENT_MODEL_*`
and `N_LINHAS_REM` are null in ~6% of rows; the per-state ingestion
notebook already documented this pattern ("no REM match") month by month:
an event's REM file may not have been processed yet in the reference month.
This is not an error in our pipeline, it is a known lag in the source data
itself. Filling with 0 would invent a reimbursement value that does not
exist (0 would mean "event with no cost", not "cost not yet reported",
which would distort any value-based analysis). We keep it null and add an
explicit indicator column, so the "null" becomes a usable signal instead of
just a hole in the base.

In [19]:
df["IND_REIMBURSEMENT_REPORTED"] = df["VL_TOTAL_REMUNERACAO"].notna().astype(int)
print(df["IND_REIMBURSEMENT_REPORTED"].value_counts())
print(f"\n{(1 - df['IND_REIMBURSEMENT_REPORTED'].mean()) * 100:.2f}% of rows still without reimbursement data processed by ANS.")

IND_REIMBURSEMENT_REPORTED
1    290637
0     19703
Name: count, dtype: int64

6.35% of rows still without reimbursement data processed by ANS.


### Group 3 — Structural absence tied to `IND_PACOTE` (keep null, already documented since ingestion)

`QT_ITEM_EVENTO_INFORMADO`, `VL_ITEM_EVENTO_INFORMADO` and
`VL_ITEM_PAGO_FORNECEDOR` are null in exactly the rows where `IND_PACOTE =
1` (item billed inside a procedure bundle, with no individual quantity/value
reported), we confirmed 100% correspondence in the real data.
We keep it null (there is no "correct" value to invent here, the item
genuinely has no individually billed quantity/value, the value is embedded
in the bundle).

In [20]:
for c in ["QT_ITEM_EVENTO_INFORMADO", "VL_ITEM_EVENTO_INFORMADO", "VL_ITEM_PAGO_FORNECEDOR"]:
    null_rate_by_bundle = df.groupby("IND_PACOTE", observed=True)[c].apply(lambda s: s.isna().mean() * 100)
    print(f"{c} -- % null by IND_PACOTE:")
    print(null_rate_by_bundle.round(2))
    print()

QT_ITEM_EVENTO_INFORMADO -- % null by IND_PACOTE:
IND_PACOTE
0      0.0
1    100.0
Name: QT_ITEM_EVENTO_INFORMADO, dtype: float64

VL_ITEM_EVENTO_INFORMADO -- % null by IND_PACOTE:
IND_PACOTE
0      0.0
1    100.0
Name: VL_ITEM_EVENTO_INFORMADO, dtype: float64



VL_ITEM_PAGO_FORNECEDOR -- % null by IND_PACOTE:
IND_PACOTE
0      0.0
1    100.0
Name: VL_ITEM_PAGO_FORNECEDOR, dtype: float64



### Group 4 — Low-percentage operational absence (impute a "not reported" label)

`REGIME_ATENDIMENTO`, `TIPO_ATENDIMENTO`, `CD_CARATER_ATENDIMENTO`,
`IND_ACIDENTE_DOENCA` and `CBO` have a low-to-moderate missing rate (between
0.1% and 44%, already excluding the 210 invalid-demographics records) and
have no single column that fully explains the absence, unlike Groups 1-3.
These are operational/categorical fields where an explicit "not reported"
label is safe (it does not invent a numeric value nor affect central-tendency
statistics) and preserves the entire row for analysis, dropping them would
cost too much volume, especially `CBO` (~44%).

In [21]:
COLUMNS_TO_IMPUTE_NOT_REPORTED = [
    "REGIME_ATENDIMENTO", "TIPO_ATENDIMENTO", "CD_CARATER_ATENDIMENTO",
    "IND_ACIDENTE_DOENCA", "CBO",
]
for c in COLUMNS_TO_IMPUTE_NOT_REPORTED:
    if "NOT_REPORTED" not in df[c].cat.categories:
        df[c] = df[c].cat.add_categories(["NOT_REPORTED"])
    df[c] = df[c].fillna("NOT_REPORTED")

print("Remaining nulls in these columns:", df[COLUMNS_TO_IMPUTE_NOT_REPORTED].isna().sum().sum())
print("\nRemaining nulls across the whole base, by column:")
print(df.isna().sum()[df.isna().sum() > 0])

Remaining nulls in these columns: 0

Remaining nulls across the whole base, by column:


QT_ITEM_EVENTO_INFORMADO    14777
VL_ITEM_EVENTO_INFORMADO    14777
VL_ITEM_PAGO_FORNECEDOR     14777
VL_TOTAL_REMUNERACAO        19703
CD_MODELO_REMUNERACAO       19703
N_LINHAS_REM                19703
dtype: int64

The only nulls remaining in the base are the Group 2 (REM, ~6%) and Group 3
(bundled item) ones, both deliberately kept as-is, with the justification
documented above and the `IND_REIMBURSEMENT_REPORTED` flag available for
downstream use.

## Phase 9 — Outliers in the monetary and quantity columns

The four continuous numeric columns of the base
(`VL_ITEM_EVENTO_INFORMADO`, `VL_ITEM_PAGO_FORNECEDOR`,
`VL_TOTAL_REMUNERACAO`, `QT_ITEM_EVENTO_INFORMADO`) are strongly
right-skewed, common for healthcare cost data.

In [22]:
MONETARY_QUANTITY_COLUMNS = [
    "VL_ITEM_EVENTO_INFORMADO", "VL_ITEM_PAGO_FORNECEDOR",
    "VL_TOTAL_REMUNERACAO", "QT_ITEM_EVENTO_INFORMADO",
]
df[MONETARY_QUANTITY_COLUMNS].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.995, 0.999])

,VL_ITEM_EVENTO_INFORMADO,VL_ITEM_PAGO_FORNECEDOR,VL_TOTAL_REMUNERACAO,QT_ITEM_EVENTO_INFORMADO
count,295563.000000,295563.000000,290637.000000,295563.000000
mean,843.891395,33.745786,842.463720,12.331470
std,5221.685290,362.751434,4409.855949,88.800396
min,0.000000,0.000000,0.000000,1.000000
50%,85.060000,0.000000,172.030000,1.000000
75%,151.870000,0.000000,326.240000,1.000000
90%,249.170000,118.090000,675.000000,1.000000
95%,870.040000,157.610000,1528.860000,60.000000
99%,23667.000000,200.000000,15000.000000,252.000000
99.5%,42587.780000,364.420000,29380.810000,420.000000


Taking `VL_TOTAL_REMUNERACAO` as an example: the median is R$ 172.03, but
the 99.9th percentile is already R$ 63,613.35 and the maximum reaches R$
157,496.59, nearly a thousand times the median. A naive outlier treatment
(e.g. removal via a 1.5×IQR rule) would cut exactly these extreme values.

**Why this is a problem here, specifically**: the business behind this
project is estimating demand for high-cost oncology treatments (targeted
therapies such as trastuzumab/T-DXd, covered by the very SBOC crosswalk we
built). An event worth R$ 60 thousand may not be noise, it may be exactly
an infusion of a state-of-the-art drug, the kind of record that matters
most to the project's partner. Aggressively removing or capping that tail
would destroy the central commercial signal of the model.

### Method chosen: log1p + capping only at statistically implausible extremes

1. We apply `log1p` (log(1+x), which handles the zero present in
   `VL_ITEM_PAGO_FORNECEDOR` cleanly) to compress the skew before any
   cutoff decision.
2. We compute the mean and standard deviation **on the log scale**.
3. We define an "implausible extreme" as any value more than 5 standard
   deviations from the mean, on the log scale, a threshold generous
   enough to preserve genuinely expensive treatments (which, even as
   extremes, still form a continuous tail rather than an isolated jump),
   and to only intervene on points that are statistically very far from
   any observed cluster.
4. We cap (replace with the limit, not remove the row) only those few
   points, preserving the event (volume count) and only adjusting the
   extreme value, and we report exactly how many points were affected,
   for full transparency of the decision.

In [23]:
def treat_outlier_log(series, k=5):
    log_series = np.log1p(series)
    mean, std = log_series.mean(), log_series.std()
    upper_bound_log = mean + k * std
    upper_bound_original = np.expm1(upper_bound_log)

    n_affected = (series > upper_bound_original).sum()
    capped_series = series.clip(upper=upper_bound_original)
    return capped_series, upper_bound_original, n_affected


outlier_report = []
for c in MONETARY_QUANTITY_COLUMNS:
    valid_series = df[c].dropna()
    capped_series, limit, n_affected = treat_outlier_log(valid_series)
    df.loc[valid_series.index, f"{c}_CAPPED"] = capped_series
    outlier_report.append({
        "column": c,
        "upper_bound_applied": round(limit, 2),
        "n_values_capped": int(n_affected),
        "pct_values_capped": round(n_affected / len(valid_series) * 100, 4),
        "original_max_value": round(valid_series.max(), 2),
    })

pd.DataFrame(outlier_report)

,column,upper_bound_applied,n_values_capped,pct_values_capped,original_max_value
0,VL_ITEM_EVENTO_INFORMADO,35068.52,2130,0.7207,224151.84
1,VL_ITEM_PAGO_FORNECEDOR,11541.49,101,0.0342,22000.00
2,VL_TOTAL_REMUNERACAO,53634.23,440,0.1514,157496.59
3,QT_ITEM_EVENTO_INFORMADO,331.61,2611,0.8834,6000.00


We keep the original columns (`VL_ITEM_EVENTO_INFORMADO`, etc.) untouched
for audit/provenance purposes, and create the `*_CAPPED` versions as the
treated version, ready for normalization in the next phase. This keeps
explicit, at any point, what was changed and by how much, no irreversible
transformation happens to the raw data.

## Phase 10 — Encoding of categorical variables

Two decision families here, depending on the cardinality (number of
distinct categories) of each column:

- **Low-cardinality nominal** (up to ~15 categories, no natural order):
  one-hot encoding. The added dimensionality is small and controlled.
- **High cardinality** (`CD_PROCEDIMENTO` with 156 values, `ID_PLANO` with
  over 3 thousand, `CD_MUNICIPIO_BENEFICIARIO` with over 1,400,
  `CD_MUNICIPIO_PRESTADOR`, `CBO`): **kept as `category`**, no one-hot.
  One-hot encoding them now would multiply the base's width by thousands of
  mostly-empty columns, and the most appropriate encoding technique for
  high cardinality (frequency, target encoding, etc.) depends on the target
  variable that we need to define. Deciding this now, with no
  target defined, would be arbitrary and could require redoing the work
  later. We document the decision and leave the column ready (`category`
  type, no nulls) for when that happens.
- **Ordinal** (`FAIXA_ETARIA`, `PORTE`): have a natural order. We encode
  them as an ordinal integer, preserving order, instead of one-hot (which
  would throw away the order information).

In [24]:
ONE_HOT_COLUMNS = [
    "SEXO", "IND_PACOTE", "CD_CARATER_ATENDIMENTO", "TIPO_ATENDIMENTO",
    "REGIME_ATENDIMENTO", "IND_ACIDENTE_DOENCA", "NM_MODALIDADE", "UF",
    "TIPO_CONSULTA", "UNIDADE_MEDIDA", "category",
]
df_encoded = pd.get_dummies(df, columns=ONE_HOT_COLUMNS, prefix=ONE_HOT_COLUMNS, dtype=int)
print(f"Shape after one-hot: {df_encoded.shape} (was {df.shape} before)")

Shape after one-hot: (310340, 111) (was (310340, 52) before)


In [25]:
# --- FAIXA_ETARIA: ordinal, following the real chronological order of the bins ---
AGE_GROUP_ORDER = [
    "Menos de 1 ano", "1 a 4 anos", "5 a 9 anos", "10 a 14 anos",
    "15 a 19 anos", "20 a 29 anos", "30 a 39 anos", "40 a 49 anos",
    "50 a 59 anos", "60 a 69 anos", "70 a 79 anos", "80 anos ou mais",
]
age_group_map = {v: i for i, v in enumerate(AGE_GROUP_ORDER)}
df_encoded["FAIXA_ETARIA_ORDINAL"] = df["FAIXA_ETARIA"].map(age_group_map)
assert df_encoded["FAIXA_ETARIA_ORDINAL"].isna().sum() == 0, (
    "FAIXA_ETARIA value outside the map -- the invalid ones should already have been removed in Phase 6."
)

# --- PORTE: ordinal, with 'SEM BENEFICIARIOS' treated as a separate flag
# (it is not 'smaller than small', it is an operator with no active
# beneficiary base in the period -- it is not part of the same size scale) ---
SIZE_ORDER = {"PEQUENO": 0, "MÉDIO": 1, "GRANDE": 2}
df_encoded["PORTE_ORDINAL"] = df["PORTE"].map(SIZE_ORDER)
df_encoded["IND_SIZE_NO_BENEFICIARIES"] = (df["PORTE"] == "SEM BENEFICIÁRIOS").astype(int)
df_encoded["PORTE_ORDINAL"] = df_encoded["PORTE_ORDINAL"].fillna(-1).astype(int)

print(df_encoded[["FAIXA_ETARIA_ORDINAL", "PORTE_ORDINAL", "IND_SIZE_NO_BENEFICIARIES"]].describe())

       FAIXA_ETARIA_ORDINAL  PORTE_ORDINAL  IND_SIZE_NO_BENEFICIARIES
count         310340.000000  310340.000000              310340.000000
mean               7.462177       1.730679                   0.003593
std                1.362805       0.550379                   0.059833
min                5.000000      -1.000000                   0.000000
25%                7.000000       2.000000                   0.000000
50%                7.000000       2.000000                   0.000000
75%                8.000000       2.000000                   0.000000
max               11.000000       2.000000                   1.000000


## Phase 11 — Normalization of numerical variables

We apply min-max scaling (0 to 1) to the `*_CAPPED` versions (already
treated in Phase 9) of the monetary/quantity columns, on the `log1p` scale, not the original scale. Two justifications:

1. **Why log before scaling, instead of min-max directly on the original
   scale**: min-max on the original scale would compress almost every
   value (most of them clustered near the median, R$ 173) toward 0, with
   the upper extreme "pulling" the entire scale, the log already fixes
   the skew before normalizing, so the final scale better preserves the
   relative position of most events, not just of the extremes.
2. **Why min-max and not z-score here**: for the project's next steps, a 0-1
   min-max scale is more interpretable for whoever reads the report. If
   the model specifically requires z-score because of the chosen
   algorithm, that is a trivial transformation to redo on the same treated
   base, not a structural rework.

In [26]:
COLUMNS_TO_NORMALIZE = [f"{c}_CAPPED" for c in MONETARY_QUANTITY_COLUMNS]

for c in COLUMNS_TO_NORMALIZE:
    log_col = np.log1p(df_encoded[c])
    minimum, maximum = log_col.min(), log_col.max()
    df_encoded[f"{c}_LOG_NORM"] = (log_col - minimum) / (maximum - minimum)

df_encoded[[f"{c}_LOG_NORM" for c in COLUMNS_TO_NORMALIZE]].describe()

,VL_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,VL_ITEM_PAGO_FORNECEDOR_CAPPED_LOG_NORM,VL_TOTAL_REMUNERACAO_CAPPED_LOG_NORM,QT_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM
count,295563.000000,295563.000000,290637.000000,295563.000000
mean,0.452905,0.072756,0.495276,0.045755
std,0.108033,0.185392,0.100707,0.184575
min,0.000000,0.000000,0.000000,0.000000
25%,0.401297,0.000000,0.438925,0.000000
50%,0.425705,0.000000,0.473231,0.000000
75%,0.480606,0.000000,0.531746,0.000000
max,1.000000,1.000000,1.000000,1.000000


## Phase 12 — Documented finding: specificity of the clinical proxy

One finding worth recording explicitly, because it quantifies a real limit
of the TUSS/SBOC proxy used in this project (with no
lab result available, the "potential patient" is inferred from a billed
procedure/drug, not from a confirmed diagnosis).

The crosswalk classifies every code by `subtype`: most `specific` codes
fall under `"general"`, cytotoxic chemotherapy drugs widely used across
several cancer types (not just breast), such as doxorubicin and
cyclophosphamide. A minority falls under more breast-specific subtypes
(HER2+, HR+/HER2-, triple-negative, BRCA-mutated).

If the proxy were perfectly breast-specific, we would expect very few male
records (male breast cancer exists, but is rare, ~1% of cases globally).
What the base actually shows:

In [27]:
sex_subtype_crosstab = pd.crosstab(df["subtype"], df["SEXO"])
sex_subtype_crosstab["pct_male"] = (
    sex_subtype_crosstab.get("Masculino", 0) /
    sex_subtype_crosstab.sum(axis=1) * 100
).round(2)
sex_subtype_crosstab.sort_values("pct_male", ascending=False)

SEXO,Feminino,Masculino,pct_male
subtype,,,
triple-negative,1288,451,25.93
general,282312,12936,4.38
BRCA-mutated,511,22,4.13
HR+/HER2-,10638,432,3.90
HER2+,1271,18,1.40
HER2+/HER2-low,428,1,0.23
HER2 undetermined,28,0,0.00
NTRK+,4,0,0.00


**Reading of the finding**: in the `"general"` subtype (broad cytotoxic
drugs), the male percentage is visibly higher than the expected incidence
of male breast cancer, direct evidence that part of these events likely
correspond to patients being treated for **other** cancer types with the
same drugs, not breast cancer. This confirms, with real data rather than
just theory.

**What we do with this now**: we do not remove these rows, discarding the
entire `"general"` subtype would cut most of the base (and, with it, the
legitimate breast-cancer cases that use these same drugs, which are the
majority). Instead, we create a proxy-confidence column, so the decision of
whether and how to weight these records differently stays available.

In [28]:
df_encoded["IND_HIGH_SPECIFICITY_PROXY"] = (df["subtype"] != "general").astype(int)
print(df_encoded["IND_HIGH_SPECIFICITY_PROXY"].value_counts(normalize=True).round(4) * 100)

IND_HIGH_SPECIFICITY_PROXY
0    95.14
1     4.86
Name: proportion, dtype: float64


## Phase 13 — Exporting the treated parquet and the decision dictionary

We export the final result and a column-by-column dictionary with the
treatment decision applied and its reasoning, the same provenance logic
already used by the per-state ingestion notebook, now extended to this
stage's cleaning/transformation decisions.

In [29]:
print(f"Final shape of the treated base: {df_encoded.shape}")
print(f"Unique events: {df['ID_EVENTO_ATENCAO_SAUDE'].nunique():,}")
print(f"States present: {sorted(df['UF'].unique())}")
print(f"Period: {df['DT_REALIZACAO'].min()} to {df['DT_REALIZACAO'].max()}")

df_encoded.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")

Final shape of the treated base: (310340, 119)
Unique events: 291,864
States present: ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO']


Period: 2025-01-01 00:00:00 to 2025-12-01 00:00:00



Saved to: C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_data_en.parquet


In [30]:
DECISIONS_DICTIONARY = [
    {"column_or_group": "TUSS/SBOC filter (Phase 1)", "decision": "Keep only 'specific' codes",
     "reason": "Discards the ambiguity of generic staging exams (no CID available to confirm breast relevance)."},
    {"column_or_group": "treatment_line, subtype, CD_MODELO_REMUNERACAO", "decision": "Decompose into indicator columns (IND_*)",
     "reason": "Multivalued columns ('a|b') are not directly usable in a model; normalization requested by the advisor."},
    {"column_or_group": "Fully duplicated rows", "decision": f"Remove ({duplicates.sum():,} rows, {duplicates.mean() * 100:.3f}%)",
     "reason": "Likely TISS resubmission/retransmission; counting twice inflates demand volume."},
    {"column_or_group": "IND_TABELA_PROPRIA", "decision": "Drop column",
     "reason": "Constant (1 unique value) -- zero variance."},
    {"column_or_group": "LG_VALOR_PREESTABELECIDO", "decision": "Drop column",
     "reason": "Near-constant (99.999% the same value)."},
    {"column_or_group": "SAUDE_OCUPACIONAL", "decision": "Drop column",
     "reason": f"{diagnostics.loc['SAUDE_OCUPACIONAL', 'pct_null']:.2f}% missing."},
    {"column_or_group": "CD_MOTIVO_SAIDA", "decision": "Drop column",
     "reason": f"{diagnostics.loc['CD_MOTIVO_SAIDA', 'pct_null']:.2f}% missing and low conceptual relevance to demand volume."},
    {"column_or_group": "CD_MODALIDADE", "decision": "Drop column",
     "reason": "Redundant with NM_MODALIDADE (1:1 correspondence); the readable version was kept."},
    {"column_or_group": f"{n_invalid_rows:,} rows with invalid FAIXA_ETARIA / null SEXO / null municipality", "decision": "Drop rows",
     "reason": "Same cluster of events with the entire demographic block missing at the source; these are core model variables, not imputable without inventing data."},
    {"column_or_group": "UNIDADE_MEDIDA", "decision": "Impute 'NOT_APPLICABLE_PROCEDURE'",
     "reason": "Absence 100% explained by CD_TABELA_REFERENCIA (only exists for drugs)."},
    {"column_or_group": "TIPO_CONSULTA", "decision": "Impute 'NOT_APPLICABLE_NOT_A_CONSULTATION'",
     "reason": "Absence explained by the nature of the event (not a consultation)."},
    {"column_or_group": "VL_TOTAL_REMUNERACAO, CD_MODELO_REMUNERACAO, N_LINHAS_REM", "decision": "Keep null + IND_REIMBURSEMENT_REPORTED flag",
     "reason": "ANS REM processing lag (~6%), documented since ingestion; 0 would invent a nonexistent cost."},
    {"column_or_group": "QT_ITEM_EVENTO_INFORMADO, VL_ITEM_EVENTO_INFORMADO, VL_ITEM_PAGO_FORNECEDOR", "decision": "Keep null",
     "reason": "100% correlated with IND_PACOTE=1 (bundled item, no individual value)."},
    {"column_or_group": "REGIME_ATENDIMENTO, TIPO_ATENDIMENTO, CD_CARATER_ATENDIMENTO, IND_ACIDENTE_DOENCA, CBO", "decision": "Impute 'NOT_REPORTED'",
     "reason": "Low/moderate operational absence with no explanatory column; dropping would cost too much volume (CBO ~44%)."},
    {"column_or_group": "VL_*/QT_* (outliers)", "decision": "log1p + capping only above 5 standard deviations on the log scale",
     "reason": "The long tail likely reflects genuinely expensive oncology treatments (the project's central commercial signal), not noise -- see Risk R03."},
    {"column_or_group": "SEXO, IND_PACOTE, CD_CARATER_ATENDIMENTO, TIPO_ATENDIMENTO, REGIME_ATENDIMENTO, IND_ACIDENTE_DOENCA, NM_MODALIDADE, UF, TIPO_CONSULTA, UNIDADE_MEDIDA, category", "decision": "One-hot encoding",
     "reason": "Low nominal cardinality, no natural order."},
    {"column_or_group": "FAIXA_ETARIA, PORTE", "decision": "Ordinal encoding",
     "reason": "Has a natural order; one-hot would discard that information."},
    {"column_or_group": "CD_PROCEDIMENTO, ID_PLANO, CD_MUNICIPIO_*, CBO", "decision": "Keep as category, no one-hot yet",
     "reason": "High cardinality; the definitive encoding (frequency/target) depends on the target variable of Card 12, not yet defined."},
    {"column_or_group": f"{under_20.sum():,} rows with FAIXA_ETARIA under 20 years old (Phase 7)", "decision": "Drop rows",
     "reason": "Breast cancer is clinically negligible before age 20; 99.93% of these rows carry the non-specific 'general' subtype (proxy specificity 0.0656% vs. 4.66% base-wide) and 41.72% are male (vs. ~1% expected even among adult cases), indicating contamination from other pediatric/adolescent malignancies (leukemia, lymphoma, sarcoma) treated with the same broad cytotoxic drugs, not real breast-cancer demand."},
]

decisions_df = pd.DataFrame(DECISIONS_DICTIONARY)
decisions_df.to_csv(DECISIONS_DICTIONARY_PATH, index=False)
print(f"Decisions dictionary saved to: {DECISIONS_DICTIONARY_PATH}")
decisions_df

Decisions dictionary saved to: C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_decisions_dictionary.csv


,column_or_group,decision,reason
0,TUSS/SBOC filter (Phase 1),Keep only 'specific' codes,Discards the ambiguity of generic staging exam...
1,"treatment_line, subtype, CD_MODELO_REMUNERACAO",Decompose into indicator columns (IND_*),Multivalued columns ('a|b') are not directly u...
2,Fully duplicated rows,"Remove (203 rows, 0.063%)",Likely TISS resubmission/retransmission; count...
3,IND_TABELA_PROPRIA,Drop column,Constant (1 unique value) -- zero variance.
4,LG_VALOR_PREESTABELECIDO,Drop column,Near-constant (99.999% the same value).
5,SAUDE_OCUPACIONAL,Drop column,99.31% missing.
6,CD_MOTIVO_SAIDA,Drop column,69.93% missing and low conceptual relevance to...
7,CD_MODALIDADE,Drop column,Redundant with NM_MODALIDADE (1:1 corresponden...
8,210 rows with invalid FAIXA_ETARIA / null SEXO...,Drop rows,Same cluster of events with the entire demogra...
9,UNIDADE_MEDIDA,Impute 'NOT_APPLICABLE_PROCEDURE',Absence 100% explained by CD_TABELA_REFERENCIA...


## Executive summary of this stage

| Step | Result |
|---|---|
| Rows in the raw North region base | 59,913,304 |
| Rows after the TUSS/SBOC filter (breast cancer) | 324,469 (0.5416%) |
| Exact duplicates removed | 203 (0.063%) |
| Rows with invalid demographics removed | 210 (0.065%) |
| Rows with age-implausibility (under 20 years old) removed | 13,716 (4.2326%) |
| **Rows in the final treated base** | 310,340 (0.5180% of the raw base) -- see the shape printed in Phase 13 |
| Columns dropped | 5 (`IND_TABELA_PROPRIA`, `LG_VALOR_PREESTABELECIDO`, `SAUDE_OCUPACIONAL`, `CD_MOTIVO_SAIDA`, `CD_MODALIDADE`) |
| New columns created | treatment-line and subtype indicators (crosswalk), reimbursement-model indicators, quality flags (`IND_REIMBURSEMENT_REPORTED`, `IND_HIGH_SPECIFICITY_PROXY`, `IND_SIZE_NO_BENEFICIARIES`), capped/normalized versions of the monetary/quantity columns |
| Output file | `data/processed/north_region_mama_2025_data_en.parquet` |
| Decisions dictionary | `data/processed/north_region_mama_2025_decisions_dictionary.csv` |

### References

1. Sociedade Brasileira de Oncologia Clinica (SBOC). SBOC 2025 Guidelines - Breast Cancer: Staging (v9), Neoadjuvant (v6), Adjuvant (v6), Advanced Disease (v12). Sao Paulo: SBOC; 2025.
2. Agencia Nacional de Saude Suplementar (ANS). Terminologia Unificada da Saude Suplementar (TUSS) and ANS Open Data - TISS Ambulatorial [Internet]. Brasilia: ANS; 2026. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/
3. Instituto Nacional de Cancer (INCA). Estimativa 2026: incidencia de cancer no Brasil [Internet]. Rio de Janeiro: INCA; 2026 [cited 2026 Aug 14]. Available from: https://www.gov.br/inca/pt-br/assuntos/cancer/numeros/estimativa/introducao